In [ ]:
# Lab type: extend
# Course: AI403 — Building Production AI Agents
# Lesson: Loop Depth, Tool Budgets, and Cost per Resolved Task
# Task: The team runs its agent with a generous 60-step cap and reports cost per run. Extend the
# harness with cost per resolved task, a data-driven step cap, and per-tool and token budgets.

# Lab: What Does a Resolved Ticket Cost?

The tickets and their tool sequences are simulated (seeded, illustrative prices), so every number reproduces. The loop and the budgets are what you'll extend.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
# No installs needed: this lab uses the Python standard library only.

In [ ]:
import random, statistics
from collections import Counter

PRICE_PER_MTOK = 3.00      # input price, dollars per million tokens (illustrative)
HUMAN_COST = 2.50          # a person handling a ticket the agent didn't resolve
RESULT_TOKENS = {"web_search": 1500, "lookup": 400}
BASE_CONTEXT = 3000

def make_tasks(n=1000, seed=7):
    """1,000 simulated tickets. Each has the sequence of tool calls the agent will make.
    About 12% can't be resolved by the agent: it keeps searching until something stops it."""
    rng = random.Random(seed)
    tasks = []
    for _ in range(n):
        if rng.random() < 0.12:
            tools = ["web_search" if rng.random() < 0.7 else "lookup" for _ in range(60)]
            tasks.append({"solvable": False, "tools": tools})
        else:
            needed = min(3 + int(rng.expovariate(0.35)), 40)
            tools = ["web_search" if rng.random() < 0.2 else "lookup" for _ in range(needed)]
            tasks.append({"solvable": True, "tools": tools})
    return tasks

TASKS = make_tasks()

def run_task(task, max_steps):
    """The baseline loop: a step cap and nothing else. Every step re-sends the whole context."""
    context, spent = BASE_CONTEXT, 0
    for step, tool in enumerate(task["tools"], 1):
        if step > max_steps:
            return {"status": "step_limit", "steps": step - 1, "tokens": spent}
        spent += context
        context += RESULT_TOKENS[tool]
    if task["solvable"]:
        return {"status": "done", "steps": len(task["tools"]), "tokens": spent + context}
    return {"status": "step_limit", "steps": len(task["tools"]), "tokens": spent}

def dollars(tokens):
    return tokens * PRICE_PER_MTOK / 1e6

print(len(TASKS), "tickets")

## The baseline report

In [ ]:
runs = [run_task(t, max_steps=60) for t in TASKS]      # "generous, so it never gives up early"
print(f"cost per run: ${statistics.mean(dollars(r['tokens']) for r in runs):.3f}")
print(Counter(r["status"] for r in runs))

## Extension 1: the unit that matters

**Question 1.** Write `cost_report(runs)` returning the resolution rate, agent cost per resolved ticket, and total cost per ticket including `HUMAN_COST` for every unresolved one. Apply it to the baseline. Where does most of the spend go?

In [ ]:
# Work here: cost_report

<details>
<summary>🔑 Reveal answer — Question 1</summary>

```python
def cost_report(runs):
    resolved = [r for r in runs if r["status"] == "done"]
    spend = sum(dollars(r["tokens"]) for r in runs)
    rate = len(resolved) / len(runs)
    return {"resolved": round(rate, 3),
            "agent_$_per_resolved": round(spend / len(resolved), 3),
            "total_$_per_ticket": round(spend / len(runs) + (1 - rate) * HUMAN_COST, 3)}

print(cost_report(runs))
failed = [r for r in runs if r["status"] != "done"]
share = sum(dollars(r["tokens"]) for r in failed) / sum(dollars(r["tokens"]) for r in runs)
print(f"share of agent spend on runs that resolved nothing: {share:.0%}")
```

Most of the agent's spend goes on the ~12% of tickets that were never going to be resolved: each one runs all 60 steps, and later steps are the most expensive because the context has grown.

</details>

## Extension 2: choose the cap from data

**Question 2.** Using the baseline runs, find the 50th, 90th, 95th and 99th percentile step counts of *successful* runs. Then sweep caps `(6, 10, 14, 18, 25, 40, 60)` with `cost_report` and pick one. Justify it in one line.

In [ ]:
# Work here: percentiles and the sweep

<details>
<summary>🔑 Reveal answer — Question 2</summary>

```python
ok = sorted(r["steps"] for r in runs if r["status"] == "done")
print("p50/p90/p95/p99:", [ok[int(q * (len(ok) - 1))] for q in (0.5, 0.9, 0.95, 0.99)])
for cap in (6, 10, 14, 18, 25, 40, 60):
    print(cap, cost_report([run_task(t, cap) for t in TASKS]))
```

Successful runs rarely need more than 16 steps (p99). Resolution flattens by about 14–18, while total cost per ticket is lowest there and more than doubles by 60. A cap of **16–18** — just above where success flattens — resolves essentially everything the agent can resolve.

</details>

## Extension 3: more budgets than steps

**Question 3.** Write `run_task_v2(task, max_steps, per_tool=None, token_budget=None)`: same loop, plus a per-tool call budget (status `tool_budget`) and a per-run token budget (status `budget_exhausted`), both checked *before* the step runs. Compare cap 18 alone, cap 18 with `per_tool={"web_search": 5}`, and both plus a token budget set at the 99th percentile of successful runs' tokens.

In [ ]:
# Work here: run_task_v2 and the comparison

<details>
<summary>🔑 Reveal answer — Question 3</summary>

```python
def run_task_v2(task, max_steps, per_tool=None, token_budget=None):
    context, spent, used = BASE_CONTEXT, 0, Counter()
    for step, tool in enumerate(task["tools"], 1):
        if step > max_steps:
            return {"status": "step_limit", "steps": step - 1, "tokens": spent}
        if per_tool and used[tool] >= per_tool.get(tool, float("inf")):
            return {"status": "tool_budget", "steps": step - 1, "tokens": spent}
        if token_budget and spent + context > token_budget:
            return {"status": "budget_exhausted", "steps": step - 1, "tokens": spent}
        spent += context
        context += RESULT_TOKENS[tool]
        used[tool] += 1
    if task["solvable"]:
        return {"status": "done", "steps": len(task["tools"]), "tokens": spent + context}
    return {"status": "step_limit", "steps": len(task["tools"]), "tokens": spent}

ok_tokens = sorted(r["tokens"] for r in runs if r["status"] == "done")
p99_tokens = ok_tokens[int(0.99 * (len(ok_tokens) - 1))]
configs = {"cap 18": {}, "cap 18 + 5 searches": {"per_tool": {"web_search": 5}},
           "cap 18 + 5 searches + p99 tokens": {"per_tool": {"web_search": 5}, "token_budget": p99_tokens}}
for name, kw in configs.items():
    print(f"{name:<34}", cost_report([run_task_v2(t, 18, **kw) for t in TASKS]))
```

The search budget stops the unresolvable tickets' search storms early, cutting total cost per ticket by about 13% (and agent cost per resolved ticket by about 40%) at essentially the same resolution. The token budget, set from the successful runs, mostly overlaps with the other two here; it earns its place when individual tool results are unexpectedly large.

</details>

**Question 4 (judgement).** Your chosen configuration goes live. What single number from the status counts would you chart daily, and what would a rise in it tell you?

In [ ]:
# Your answer (as a comment):

<details>
<summary>🔑 Reveal answer — Question 4</summary>

Chart the **share of runs ending in a budget exit** (`step_limit` + `tool_budget` + `budget_exhausted`), ideally per exit type. It's stable when traffic and tools are stable. A rise means something changed upstream — a new kind of ticket the agent can't resolve, a tool that started failing or returning larger results, or a search index gone stale — and it usually shows up before anyone complains. Every breach should also be logged with its ticket so the new cases can join the labelled eval set.

</details>

## Summary

1. The unit that matters is cost per _______ task, including human handling of failures.
2. Choose the step cap from the step counts of _______ runs.
3. Expensive tools get their own per-tool _______, enforced in the loop.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **resolved**
2. **successful**
3. **budget**

</details>